# Chiến thuật hiện tại — SHORT phá dải Bollinger sau giai đoạn biến động thấp, trailing 1.5R, danh mục nhiều coin

Dùng `vol_filter_engine.py` cùng thư mục (numba CPU). Sửa **ô Cấu hình**, rồi **Run All**.

**Luật (15min, mỗi coin một lệnh mỗi lúc):**
- **Vào short** ở close nến `t` khi `close[t-1] > lower[t] > close[t]` (đóng cửa phá xuống dải dưới Bollinger) **và** `vol_ratio[t] < vol_ratio_max` — `vol_ratio` = độ lệch chuẩn log-return `vol_window` nến gần nhất / trung vị của nó trong `vol_days` ngày (thị trường đang yên hơn bình thường). **Không dùng RSI** (`rsi_threshold = 100`).
- **Rủi ro (R):** `sl_multiplier × (upper − entry)`; stop ban đầu = entry + R.
- **Thoát:** trailing — sau mỗi nến, stop kéo xuống `đáy thấp nhất từ lúc vào + trail_r × R` (không nới ra); không chốt lời; timeout `max_holding_bars` nến. Phí vào `0.1% / (R / giá)`, phí giữ mỗi 8 giờ.

**Chọn coin (xét lại mỗi `REBALANCE_MONTHS` tháng, chỉ dùng dữ liệu trước kỳ):** bỏ `EXCLUDE` (CRV, TRX — lỗ ở mọi lần kiểm tra) và `LOW_VOL_FRACTION` coin có biến động năm hóa thấp nhất trong `VOL_LOOKBACK_DAYS` ngày trước kỳ (dải hẹp → stop hẹp → phí chiếm 0.06–0.09R/lệnh, gấp đôi coin biến động cao).

**Danh mục:** chia đều cho các coin được chọn trong kỳ — mỗi lệnh rủi ro `1 / số coin` R của danh mục; R danh mục = trung bình R mỗi coin.

**Đọc kết quả cho đúng (nghiên cứu 24–27/10/2026, 36–47 coin):**
- `PARAMS` mặc định được chọn từ vùng tốt nhất của grid trên **2022–2025** → backtest 2022–2025 ở ô 5 là **trong mẫu** (lạc quan). Chỉ 2026 là ngoài mẫu với bộ này.
- Ô 6 (walk-forward) là **ngoài mẫu**: mỗi lượt chọn tham số chỉ bằng dữ liệu trước đó. Nghiên cứu: 24/6, một bộ chung, 2023-01 → 2026-10 ≈ **+5–6 R/coin/năm**, 7/8 lượt lãi; 2026 yếu (≈ +1 R/coin nửa đầu năm, ≈ 0 bốn tháng sau).
- Chọn **một bộ chung** cho mọi coin ổn định hơn chọn riêng từng coin. Coin biến động thấp nhất (đo trước) lãi ngoài mẫu ~+4.1 so với ~+6.6 R/năm của phần còn lại.
- Nhiều lệnh short mở cùng lúc khi thị trường sập (tối đa ~34/36 coin) — rủi ro tương quan; xem ô 5.

In [ ]:
# CẤU HÌNH
DATA_DIR = "../data_5m"
TIMEFRAME = "15min"
DATA_START = "2021-01-01"        # dữ liệu 5m có từ 2021-01
DATA_END = "2026-10-19"
BACKTEST_START = "2022-01-01"    # cần ≥ VOL_LOOKBACK_DAYS dữ liệu trước kỳ đầu để chọn coin

PARAMS = {                       # bộ cố định (chọn trên 2022–2025: ô 5 là trong mẫu)
    "boll_window": 200, "boll_window_dev": 2.0, "sl_multiplier": 0.6,
    "vol_window": 288, "vol_days": 90, "vol_ratio_max": 0.9,
    "trail_r": 1.5, "tp_sl_ratio": 0, "max_holding_bars": 384,
    "rsi_window": 5, "rsi_threshold": 100,   # 100 = không dùng RSI
}

COINS = "all"                    # "all" = mọi file trong DATA_DIR, hoặc danh sách ["avax", "doge", ...]
EXCLUDE = ["crv", "trx"]
LOW_VOL_FRACTION = 0.25          # bỏ 25% coin biến động thấp nhất mỗi kỳ; 0 = không lọc
VOL_LOOKBACK_DAYS = 365
REBALANCE_MONTHS = 6

# Walk-forward (ô 6): chọn lại tham số mỗi WF_TEST tháng bằng WF_TRAIN tháng trước đó
RUN_WALK_FORWARD = True
WF_TRAIN, WF_TEST = 24, 6        # hoặc 36, 12
WF_FIRST_TEST = "2023-01-01"     # ≥ DATA_START + WF_TRAIN tháng
WF_SELECT = "chung"              # "chung" = 1 top-K chung cho mọi coin (khuyên dùng), "riêng" = từng coin
WF_TOP_K = 10
WF_MIN_TRADES_PER_YEAR = 10
WF_GRID = {                      # 2 160 bộ; ~2–3 phút cho ~45 coin
    "boll_window": [80, 100, 150, 200, 300], "boll_window_dev": [1.8, 2.0, 2.2],
    "sl_multiplier": [0.5, 0.6, 0.8, 1.0], "vol_window": [96, 288], "vol_days": [30, 90],
    "vol_ratio_max": [0.8, 0.9, 1.0], "trail_r": [1.0, 1.5, 2.0],
    "tp_sl_ratio": [0], "max_holding_bars": [384], "rsi_window": [5], "rsi_threshold": [100],
}

SHOW_COIN = "avax"               # ô 7: biểu đồ từng lệnh của coin này

In [ ]:
# Nạp vol_filter_engine.py cùng thư mục
from pathlib import Path
import importlib.util
import sys
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
FOLDER = "short_bollinger+RSI+volfilter"
candidates = []
for parent in (Path.cwd(), *Path.cwd().parents):
    if parent.name == FOLDER:
        candidates.append(parent / "vol_filter_engine.py")
    candidates.append(parent / "reference" / FOLDER / "vol_filter_engine.py")
engine_path = next((p.resolve() for p in candidates if p.is_file()), None)
if engine_path is None:
    raise FileNotFoundError("Không tìm thấy vol_filter_engine.py")
for name in ("vol_filter_engine", "multi_position_engine"):
    sys.modules.pop(name, None)
spec = importlib.util.spec_from_file_location("vol_filter_engine", engine_path)
vfe = importlib.util.module_from_spec(spec)
sys.modules["vol_filter_engine"] = vfe
spec.loader.exec_module(vfe)
data_dir = (engine_path.parent / DATA_DIR).resolve() if not Path(DATA_DIR).is_absolute() else Path(DATA_DIR)
print("Engine:", engine_path)
print("Dữ liệu:", data_dir)

In [ ]:
# Đọc dữ liệu mọi coin
if COINS == "all":
    coin_list = sorted(p.name.replace("usdt_futures_5min.csv", "") for p in data_dir.glob("*usdt_futures_5min.csv"))
else:
    coin_list = list(COINS)
coin_list = [c for c in coin_list if c not in EXCLUDE]
bars, daily = {}, {}
t0 = time.perf_counter()
for coin in coin_list:
    try:
        b = vfe.load_bars(data_dir / f"{coin}usdt_futures_5min.csv", TIMEFRAME, DATA_START, DATA_END)
    except (FileNotFoundError, ValueError) as exc:
        print(coin, "bỏ qua:", exc)
        continue
    bars[coin] = b
    daily[coin] = b.set_index("timestamp").close.resample("D").last().dropna()
print(f"{len(bars)} coin ({time.perf_counter() - t0:.0f}s); bỏ theo EXCLUDE: {EXCLUDE}")
display(pd.DataFrame({c: {"từ": b.timestamp.iloc[0].date(), "đến": b.timestamp.iloc[-1].date(), "nến": len(b)}
                      for c, b in bars.items()}).T.sort_values("từ").tail(12))

In [ ]:
# Chọn coin mỗi kỳ: đủ VOL_LOOKBACK_DAYS dữ liệu trước kỳ, bỏ LOW_VOL_FRACTION coin biến động thấp nhất
periods = pd.date_range(BACKTEST_START, DATA_END, freq=f"{REBALANCE_MONTHS}MS")


def trailing_vol(coin, start):
    d = daily[coin]
    w = d[(d.index >= start - pd.Timedelta(days=VOL_LOOKBACK_DAYS)) & (d.index < start)]
    if len(w) < 0.9 * VOL_LOOKBACK_DAYS or d.index[-1] < start:
        return np.nan
    return np.log(w).diff().std() * np.sqrt(365)


def selected_coins(start):
    vols = pd.Series({c: trailing_vol(c, start) for c in bars}).dropna()
    if LOW_VOL_FRACTION > 0 and len(vols):
        vols = vols[vols > vols.quantile(LOW_VOL_FRACTION)]
    return vols


selection = {p: selected_coins(p) for p in periods}
rows = []
for p, vols in selection.items():
    have = pd.Series({c: trailing_vol(c, p) for c in bars}).dropna()
    rows.append({"kỳ": p.date(), "số coin": len(vols), "ngưỡng vol": have.quantile(LOW_VOL_FRACTION) if LOW_VOL_FRACTION > 0 else 0,
                 "bị loại (vol thấp)": " ".join(sorted(set(have.index) - set(vols.index)))})
display(pd.DataFrame(rows).round(2))

In [ ]:
# Backtest bộ PARAMS cố định cho danh mục (lệnh của coin chỉ tính khi coin được chọn ở kỳ vào lệnh)
t0 = time.perf_counter()
parts = []
for coin, b in bars.items():
    tr = vfe.trace(vfe.SIDE_SHORT, b, PARAMS, TIMEFRAME)
    parts.append(tr.assign(coin=coin))
trades = pd.concat(parts, ignore_index=True)
trades["entry_ts"] = pd.to_datetime(trades.entry_timestamp)
trades["exit_ts"] = pd.to_datetime(trades.exit_timestamp)
trades = trades[trades.entry_ts >= pd.Timestamp(BACKTEST_START)].copy()
trades["kỳ"] = periods[np.searchsorted(periods.values, trades.entry_ts.values, side="right") - 1]
n_sel = {p: len(v) for p, v in selection.items()}
trades["chọn"] = [c in selection[p].index for c, p in zip(trades.coin, trades["kỳ"])]
trades["trọng_số"] = [1 / n_sel[p] if n_sel[p] else 0 for p in trades["kỳ"]]
print(f"{len(trades):,} lệnh ({trades.chọn.sum():,} ở coin được chọn), {time.perf_counter() - t0:.0f}s")


def portfolio(tr, label):
    days = pd.date_range(BACKTEST_START, DATA_END, freq="D")
    d = (tr.net_r * tr.trọng_số).groupby(tr.exit_ts.dt.normalize()).sum().reindex(days, fill_value=0.0)
    eq = d.cumsum()
    dd = np.maximum.accumulate(np.maximum(eq, 0)) - eq
    m = d.groupby(d.index.to_period("M")).sum()
    row = {"danh mục": label, **{str(y): v for y, v in d.groupby(d.index.year).sum().items()},
           "tổng R": eq.iloc[-1], "maxDD (ngày)": dd.max(), "% tháng lãi": (m > 0).mean(),
           "6 tháng tệ nhất": m.rolling(6).sum().min(), "Sharpe (năm hóa)": d.mean() / d.std() * np.sqrt(365)}
    return row, eq, dd, m


sel = trades[trades.chọn]
all_w = trades.assign(trọng_số=1 / trades.coin.nunique())
row_sel, eq_sel, dd_sel, m_sel = portfolio(sel, "coin được chọn")
row_all, eq_all, _, _ = portfolio(all_w, "mọi coin (không lọc coin)")
print("R của danh mục (mỗi lệnh rủi ro 1/số coin R):")
display(pd.DataFrame([row_sel, row_all]).round(2))
ev = pd.concat([pd.Series(1, index=sel.entry_ts), pd.Series(-1, index=sel.exit_ts)]).sort_index().cumsum()
print(f"Vị thế mở cùng lúc (coin được chọn): trung bình {ev.resample('h').last().ffill().mean():.1f}, tối đa {ev.max()}")

fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
axes[0].plot(eq_sel.index, eq_sel.to_numpy(), label="coin được chọn")
axes[0].plot(eq_all.index, eq_all.to_numpy(), label="mọi coin", alpha=0.6)
axes[0].set(title="Equity danh mục (R)", ylabel="R")
axes[0].grid(alpha=0.3)
axes[0].legend()
axes[1].fill_between(dd_sel.index, -dd_sel.to_numpy(), 0, color="tab:red", alpha=0.4)
axes[1].set(title="Drawdown (coin được chọn)", ylabel="R")
plt.tight_layout()
plt.show()

per = sel.pivot_table(index="coin", columns=sel.exit_ts.dt.year, values="net_r", aggfunc="sum").fillna(0)
per["tổng"] = per.sum(1)
per["lệnh"] = sel.groupby("coin").size()
per["tỷ lệ thắng"] = sel.groupby("coin").net_r.apply(lambda s: (s > 0).mean())
print("R theo coin và năm (R của từng coin, chưa chia trọng số; chỉ kỳ coin được chọn):")
display(per.sort_values("tổng", ascending=False).round(2))
display(sel.groupby("outcome").net_r.agg(["size", "mean", "sum"]).round(3))
print(f"Giờ giữ lệnh: trung bình {sel.holding_bars.mean() * pd.Timedelta(TIMEFRAME) / pd.Timedelta(hours=1):.1f}")
print("R danh mục theo tháng, năm gần nhất:", " ".join(f"{p.strftime('%m')}:{r:+.2f}" for p, r in
                                                        m_sel[m_sel.index.year == m_sel.index.year.max()].items()))

In [ ]:
# Walk-forward: mỗi WF_TEST tháng chọn lại top-K bộ của WF_GRID bằng perf / maxDD (theo tháng) của WF_TRAIN tháng trước,
# chỉ trên các coin được chọn ở kỳ test (cùng luật EXCLUDE + biến động, đo trước kỳ)
if RUN_WALK_FORWARD:
    wf_grid = vfe.parameter_grid(WF_GRID)
    month0 = pd.Timestamp(DATA_START).to_period("M")
    n_months = (pd.Timestamp(DATA_END).to_period("M") - month0).n + 1
    t0 = time.perf_counter()
    R_wf, N_wf, first_m = {}, {}, {}
    for coin, b in bars.items():
        R, N, _ = vfe.run_grid(vfe.SIDE_SHORT, b, wf_grid, TIMEFRAME, vfe.month_index(b, str(month0)))
        R_wf[coin] = np.pad(R, ((0, 0), (0, max(n_months - R.shape[1], 0))))[:, :n_months]
        N_wf[coin] = np.pad(N, ((0, 0), (0, max(n_months - N.shape[1], 0))))[:, :n_months]
        fts = b.timestamp.iloc[0]
        first_m[coin] = (fts.to_period("M") - month0).n + (1 if fts.day > 3 else 0)
    print(f"Grid {len(wf_grid):,} bộ × {len(bars)} coin: {time.perf_counter() - t0:.0f}s")

    starts = range((pd.Timestamp(WF_FIRST_TEST).to_period("M") - month0).n, n_months, WF_TEST)
    monthly = pd.Series(0.0, index=pd.period_range(WF_FIRST_TEST, DATA_END, freq="M"))
    fold_rows, coin_rows = [], []
    for c in starts:
        a, e = c - WF_TRAIN, min(c + WF_TEST, n_months)
        test_start = (month0 + c).to_timestamp()
        vols = selected_coins(test_start)
        members = [k for k in vols.index if first_m[k] <= a]
        if not members:
            continue
        SC, TE = [], []
        for k in members:
            x = R_wf[k][:, a:c]
            ok = N_wf[k][:, a:c].sum(1) >= WF_MIN_TRADES_PER_YEAR * WF_TRAIN / 12
            SC.append(np.where(ok, x.sum(1) / np.maximum(vfe.monthly_max_dd(x), 1.0), -np.inf))
            TE.append(R_wf[k][:, c:e])
        SC, TE = np.stack(SC), np.stack(TE)            # coins x sets, coins x sets x months
        if WF_SELECT == "chung":
            valid = np.isfinite(SC)
            mean_sc = np.where(valid.mean(0) >= 0.9, np.nanmean(np.where(valid, SC, np.nan), axis=0), -np.inf)
            tops = [np.argsort(-mean_sc)[:WF_TOP_K]] * len(members)
        else:
            tops = [np.argsort(-SC[i])[:WF_TOP_K] for i in range(len(members))]
        per_coin = np.stack([TE[i][tops[i]].mean(0) for i in range(len(members))])   # coins x months
        port = per_coin.mean(0)
        monthly.loc[[month0 + m for m in range(c, e)]] = port
        best = wf_grid.loc[tops[0][0], ["boll_window", "boll_window_dev", "sl_multiplier", "vol_window", "vol_days",
                                        "vol_ratio_max", "trail_r"]].to_dict()
        fold_rows.append({"test từ": test_start.date(), "tháng": e - c, "số coin": len(members),
                          "R/coin": port.sum(), "% coin lãi": (per_coin.sum(1) > 0).mean(),
                          "bộ #1" + (" (chung)" if WF_SELECT == "chung" else f" ({members[0]})"): best})
        coin_rows += [{"coin": k, "test từ": test_start.date(), "R": per_coin[i].sum(), "tháng": e - c}
                      for i, k in enumerate(members)]
    folds = pd.DataFrame(fold_rows)
    display(folds.round(2))
    eq = monthly.cumsum()
    print(f"Ngoài mẫu {WF_FIRST_TEST} → {DATA_END}: tổng {eq.iloc[-1]:.2f} R/coin "
          f"({eq.iloc[-1] / (len(monthly) / 12):.2f} R/coin/năm); lượt lãi {(folds['R/coin'] > 0).mean():.0%}; "
          f"maxDD (tháng) {(np.maximum.accumulate(np.maximum(eq, 0)) - eq).max():.2f}; % tháng lãi {(monthly > 0).mean():.0%}")
    print("Theo năm:", " | ".join(f"{y}: {r:+.2f}" for y, r in monthly.groupby(monthly.index.year).sum().items()))
    ax = eq.to_timestamp().plot(figsize=(14, 4), drawstyle="steps-post")
    ax.set(title=f"Walk-forward {WF_TRAIN}/{WF_TEST} tháng, top-{WF_TOP_K} {WF_SELECT} (R/coin, ngoài mẫu)", ylabel="R")
    ax.grid(alpha=0.3)
    plt.show()
    cr = pd.DataFrame(coin_rows)
    oos = cr.groupby("coin").agg(R=("R", "sum"), tháng=("tháng", "sum"), lượt=("R", "size"),
                                 lượt_lãi=("R", lambda s: (s > 0).mean()))
    oos["R/năm"] = oos.R / (oos.tháng / 12)
    print("Ngoài mẫu theo coin (chỉ các lượt coin được chọn):")
    display(oos.sort_values("R/năm", ascending=False).round(2))

In [ ]:
# Từng lệnh của SHOW_COIN (bộ PARAMS cố định): giá, điểm vào / ra, equity
tr = trades[trades.coin == SHOW_COIN].copy()
b = bars[SHOW_COIN].set_index("timestamp").close.loc[BACKTEST_START:]
print(f"{SHOW_COIN}: {len(tr)} lệnh, tổng {tr.net_r.sum():.1f} R, tỷ lệ thắng {(tr.net_r > 0).mean():.0%}")
display(tr[["entry_ts", "exit_ts", "entry_price", "risk", "stop_loss", "outcome", "holding_bars", "net_r", "chọn"]].tail(15)
        .round(4))
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
axes[0].plot(b.index, b.to_numpy(), lw=0.5, color="gray")
win, loss = tr[tr.net_r > 0], tr[tr.net_r <= 0]
axes[0].scatter(tr.entry_ts, tr.entry_price, marker="v", s=12, color="black", label="vào short")
axes[0].scatter(win.exit_ts, b.reindex(win.exit_ts, method="nearest").to_numpy(), s=10, color="tab:green", label="ra lãi")
axes[0].scatter(loss.exit_ts, b.reindex(loss.exit_ts, method="nearest").to_numpy(), s=10, color="tab:red", label="ra lỗ")
axes[0].set(title=f"{SHOW_COIN.upper()} — giá và lệnh", ylabel="USDT")
axes[0].legend()
e = tr.set_index("exit_ts").net_r.cumsum()
axes[1].step(e.index, e.to_numpy(), where="post")
axes[1].set(title="Equity của coin (R)", ylabel="R")
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()